In [0]:
%sql
drop table if exists wx_dbxproject.metadata.tables;
drop table if exists wx_dbxproject.metadata.table_parameters;
drop table if exists wx_dbxproject.metadata.table_watermarks;
drop table if exists wx_dbxproject.metadata.pipeline_runs

In [0]:
%sql
-- =====================================================
-- CATALOG & SCHEMA
-- =====================================================

CREATE CATALOG IF NOT EXISTS wx_dbxproject;

CREATE SCHEMA IF NOT EXISTS wx_dbxproject.metadata;


-- =====================================================
-- 1️⃣ metadata.tables
-- Static registry of logical tables
-- =====================================================

CREATE TABLE IF NOT EXISTS wx_dbxproject.metadata.tables (
    table_id            INT,
    table_name          STRING,
    source_system       STRING,        -- sqlserver / blob
    source_schema       STRING,        -- dbo (null for blob)
    source_table        STRING,        -- table name (null for blob)
    source_path         STRING,        -- blob path (null for sqlserver)
    target_layer        STRING,        -- silver/gold
    bronze_schema       STRING,        -- bronze
    silver_schema       STRING,        -- silver
    gold_schema         STRING,        -- gold
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
)
USING DELTA;


-- =====================================================
-- 2️⃣ metadata.table_parameters
-- Processing configuration (load type, PK, watermark)
-- =====================================================

CREATE TABLE IF NOT EXISTS wx_dbxproject.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA;


-- =====================================================
-- 3️⃣ metadata.table_watermarks
-- Stores last successful watermark per table
-- =====================================================

CREATE TABLE IF NOT EXISTS wx_dbxproject.metadata.table_watermarks (
    table_id                INT,
    last_watermark_value    STRING,     -- flexible type storage
    last_updated_at         TIMESTAMP,
    last_run_id             BIGINT
)
USING DELTA
PARTITIONED BY (table_id);


-- =====================================================
-- 4️⃣ metadata.pipeline_runs
-- Execution audit table
-- =====================================================

CREATE TABLE IF NOT EXISTS wx_dbxproject.metadata.pipeline_runs (
    run_id              BIGINT,
    table_id            INT,
    layer               STRING,        -- Bronze / Silver / Gold
    start_time          TIMESTAMP,
    end_time            TIMESTAMP,
    status              STRING,        -- SUCCESS / FAILED
    number_of_records     BIGINT,
    error_message       STRING
)
USING DELTA
PARTITIONED BY (table_id);


In [0]:
%sql
INSERT INTO wx_dbxproject.metadata.tables VALUES
(1, 'customers', 'sqlserver', 'dbo', 'customers', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),
(2, 'restaurants', 'sqlserver', 'dbo', 'restaurants', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),
(3, 'menu_items', 'sqlserver', 'dbo', 'menu_items', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),
(4, 'reviews', 'sqlserver', 'dbo', 'reviews', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),
(5, 'historical_orders', 'sqlserver', 'dbo', 'historical_orders', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 5, current_timestamp())

In [0]:
%sql
INSERT INTO wx_dbxproject.metadata.table_parameters VALUES
-- ================= customers =================
(1, 'load_type', 'FULL', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
-- ================= restaurants =================
(2, 'load_type', 'FULL', current_timestamp()),
(2, 'primary_key', 'restaurant_id', current_timestamp()),
-- ================= menu_items =================
(3, 'load_type', 'FULL', current_timestamp()),
(3, 'primary_key', 'item_id', current_timestamp()),
-- ================= reviews =================
(4, 'load_type', 'APPEND', current_timestamp()),
(4, 'primary_key', 'review_id', current_timestamp()),
(4, 'watermark_column', 'review_timestamp', current_timestamp()),
-- ================= historical_orders =================
(5, 'load_type', 'APPEND', current_timestamp()),
(5, 'primary_key', 'order_id', current_timestamp()),
(5, 'watermark_column', 'order_timestamp', current_timestamp());

In [0]:
%sql
INSERT INTO banking.metadata.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(4, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL);


In [0]:
%sql
select * from wx_dbxproject.metadata.table_parameters